In [2]:
import os
import json
import argparse
import unicodedata

import logging
import torch
import librosa
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import List, Dict, Union

from sklearn.model_selection import train_test_split
from transformers import (
Wav2Vec2CTCTokenizer,
Wav2Vec2FeatureExtractor,
Wav2Vec2Processor,
WavLMForCTC,
TrainingArguments,
Trainer,
TrainerCallback
)
import evaluate
from torch.utils.data import Dataset
import torch.nn as nn


class CommonVoicePhonemeDataset(Dataset):
    def __init__(self, dataframe, audio_dir, processor):
        self.df = dataframe.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio_path = os.path.join(self.audio_dir, row["path"])

        try:
            waveform, sr = librosa.load(audio_path, sr=16000)
            MAX_DURATION = 10  # seconds
            max_len = int(sr * MAX_DURATION)
            
            if len(waveform) > max_len:
                waveform = waveform[:max_len]

            waveform = torch.tensor(waveform)
            
        except Exception as e:
            print(f"Error loading {audio_path}: {e}")
            # Return a dummy sample in case of error
            waveform = torch.zeros(16000)

        input_values = self.processor(
            waveform,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_values.squeeze()

        tokens = row["phonemes"].split()
        labels = [
            self.processor.tokenizer.convert_tokens_to_ids(t)
            for t in tokens
        ]

        return {
            "input_values": input_values,
            "labels": torch.tensor(labels, dtype=torch.long)
        }

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id,
            -100
        )

        batch["labels"] = labels
        return batch


# =========================
# Metrics (PER)
# =========================

wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids)

    # Decode labels
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    # 🔥 REMOVE SPACES (critical)
    pred_str = [s.replace(" ", "") for s in pred_str]
    label_str = [s.replace(" ", "") for s in label_str]

    per = cer_metric.compute(predictions=pred_str, references=label_str)

    # Extra diagnostics
    avg_unique = np.mean([len(set(seq)) for seq in pred_ids])

    blank_ratio = np.mean([
        np.sum(seq == processor.tokenizer.pad_token_id) / len(seq)
        for seq in pred_ids
    ])

    return {
        "per": per,
        "avg_unique_phonemes": avg_unique,
        "blank_ratio": blank_ratio
    }


# =========================
# Safety Callback
# =========================

class SafetyCallback(TrainerCallback):
    def on_backward_end(self, args, state, control, model=None, **kwargs):
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2

        total_norm = total_norm ** 0.5
        print(f"Step {state.global_step}: Gradient norm = {total_norm:.6f}")



/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/home/imbenamor/miniconda3/envs/venv-t

In [ ]:
!

In [3]:
import torchaudio
import os

csv_path = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/commonvoice_fr_wavlm.csv"
audio_dir= "/vol/corpora/CommonVoice/cv-corpus-19.0-2024-09-13/fr/clips"
df = pd.read_csv(csv_path)
# Normalize Unicode
df["phonemes"] = df["phonemes"].apply(lambda x: unicodedata.normalize("NFC", x))


df["phonemes"] = df["phonemes"].str.replace("ː", "", regex=False)

df["phonemes"] = df["phonemes"].str.replace("dʒ", "ʒ", regex=False)

# Replace standalone ɑ but NOT ɑ̃
#df["phonemes"] = df["phonemes"].str.replace(r"\bɑ\b", "a", regex=True)

# IMPORTANT: Ensure nasal vowels are normalized to precomposed forms
df["phonemes"] = df["phonemes"].str.replace("ã", "ɑ̃")
df["phonemes"] = df["phonemes"].str.replace("ẽ", "ɛ̃")
df["phonemes"] = df["phonemes"].str.replace("õ", "ɔ̃")
df["phonemes"] = df["phonemes"].str.replace("œ̃", "ɛ̃")
df["phonemes"] = df["phonemes"].str.replace('ɹ', "ʁ")
df["phonemes"] = df["phonemes"].str.replace(r"\s+", " ", regex=True).str.strip()
# Clean spaces
df["phonemes"] = df["phonemes"].str.replace(r"\s+", " ", regex=True)
df["phonemes"] = df["phonemes"].str.strip()

# Remove any rows with missing or invalid data
initial_len = len(df)
df = df.dropna(subset=['path', 'phonemes'])
df = df[df['phonemes'].str.strip() != '']
df

,path,sentence,phonemes
0,common_voice_fr_26983275.mp3,"Depuis sa création, le réseau connaît une fréq...",d ə p y i | s a | k ʁ e a s j ɔ̃ | l ə | ʁ e z...
1,common_voice_fr_19333687.mp3,Il occupe actuellement le poste d'arrière laté...,i l | ɔ k y p | a k t y ɛ l m ɑ̃ | l ə | p ɔ s...
2,common_voice_fr_17616737.mp3,Tout a tourné contre nous.,t u t | a | t u ʁ n e | k ɔ̃ t ʁ | n u
3,common_voice_fr_23984073.mp3,Personne n'a vu passer les deux disparus.,p ɛ ʁ s ɔ n | n a | v y | p a s e | l e | d ø ...
4,common_voice_fr_19744770.mp3,Selon la légende les premiers Tchèques s'y ins...,s ə l ɔ̃ | l a | l e ʒ ɑ̃ d | l e | p ʁ ə m j ...
...,...,...,...
37777,common_voice_fr_25427642.mp3,Cette dernière refusa sous l'influence de Cléon.,s ɛ t | d ɛ ʁ n j ɛ ʁ | ʁ ə f y z a | s u | l ...
37778,common_voice_fr_33032599.mp3,Accessible au public et baignade gratuite avec...,a k s ɛ s i b l | o | p y b l i k | e | b ɛ n ...
37779,common_voice_fr_19632751.mp3,"Environ l'ont fréquenté, contre deux ans aupar...",ɑ̃ v i ʁ ɔ̃ | l ɔ̃ | f ʁ e k ɑ̃ t e | k ɔ̃ t ʁ...
37780,common_voice_fr_18510687.mp3,"Quel est l’avis du Gouvernement ? Défavorable,...",k ɛ l | ɛ | l a v i | d y | ɡ u v ɛ ʁ n ə m ɑ̃...


In [4]:
print("Splitting train/validation...")
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")

# Tokenizer
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    pad_token="[PAD]",
    word_delimiter_token="|",
    do_lower_case=False,
    replace_word_delimiter_char="|",
)

print(f"Tokenizer vocab size: {len(tokenizer)}")


feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)


# Verify phoneme tokenization
sample_phonemes = train_df['phonemes'].iloc[0]
print(f"Sample phonemes: {sample_phonemes}")
tokens = tokenizer(sample_phonemes).input_ids
print(f"Tokenized: {tokens}")
decoded = tokenizer.decode(tokens)
print(f"Decoded: {decoded}")


model = WavLMForCTC.from_pretrained(
    "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wavlm-base-plus",
    vocab_size=37,
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
)

# Freeze encoder
for param in model.wavlm.parameters():
    param.requires_grad = False

print(f"Model vocab size: {model.config.vocab_size}")


# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params:,}")


# Datasets
print("Creating datasets...")
train_dataset = CommonVoicePhonemeDataset(
    train_df,
    audio_dir,
    processor
)

val_dataset = CommonVoicePhonemeDataset(
    val_df,
    audio_dir,
    processor
)

data_collator = DataCollatorCTCWithPadding(processor=processor)

# Test a sample batch
print("Testing data loading...")
test_loader = DataLoader(train_dataset, batch_size=2, collate_fn=data_collator)
sample_batch = next(iter(test_loader))
print(f"Sample batch - Input shape: {sample_batch['input_values'].shape}, Labels shape: {sample_batch['labels'].shape}")
print(f"Input values stats: min={sample_batch['input_values'].min():.4f}, "
            f"max={sample_batch['input_values'].max():.4f}, "
            f"mean={sample_batch['input_values'].mean():.4f}")

# Check for data anomalies
if torch.isnan(sample_batch['input_values']).any():
    raise ValueError("NaN detected in input values!")
if (sample_batch['labels'] < -100).any():
    raise ValueError("Invalid label values detected!")

training_args = TrainingArguments(
        output_dir="./",

        # Batch size
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=2,

        # Training
        num_train_epochs=3,

        # Learning rate
        learning_rate=1e-3,
        warmup_steps=0,
        #lr_scheduler_type="cosine",

        # Gradient clipping
        max_grad_norm=1.0,

        # Mixed precision
        fp16=True,
        bf16=False,

        # Evaluation & saving
        evaluation_strategy="steps",
        save_strategy="steps",
        logging_steps=100,
        eval_steps=500,
        save_steps=500,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="per",
        greater_is_better=False,

        # Regularization
        weight_decay=0.0,

        # Data loading
        dataloader_num_workers=4,
        dataloader_pin_memory=True,


    )


# Trainer
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        tokenizer=processor,
        compute_metrics=compute_metrics,
        callbacks=[SafetyCallback()],
    )

trainer.train()



print("=" * 70)
print("Training complete!")
print("=" * 70)

Splitting train/validation...
Train size: 34003
Validation size: 3779
Tokenizer vocab size: 40
Sample phonemes: b ɔ ʁ d o | m e t ʁ o p ɔ l | s ɔ k y p | d ə | l a | ʒ ɛ s t j ɔ̃ | d ə | s ə | k u ʁ | d o
Tokenized: [1, 20, 25, 20, 32, 20, 2, 20, 11, 20, 20, 20, 9, 20, 3, 20, 14, 20, 32, 20, 11, 20, 12, 20, 25, 20, 8, 20, 20, 20, 13, 20, 25, 20, 7, 20, 18, 20, 12, 20, 20, 20, 2, 20, 27, 20, 20, 20, 8, 20, 0, 20, 20, 20, 34, 20, 28, 20, 13, 20, 14, 20, 6, 26, 20, 20, 2, 20, 27, 20, 20, 20, 13, 20, 27, 20, 20, 20, 7, 20, 15, 20, 32, 20, 20, 20, 2, 20, 11]
Decoded: b|ɔ|ʁ|d|o|m|e|t|ʁ|o|p|ɔ|l|s|ɔ|k|y|p|d|ə|l|a|ʒ|ɛ|s|t|jɔ̃|d|ə|s|ə|k|u|ʁ|d|o


/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of WavLMForCTC were not initialized from the model checkpoint at /vol/experiments3/imbenamor/TAPAS-FRAIS/models/wavlm-base-plus and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model vocab size: 37
Trainable parameters: 28,453 (0.03%)
Total parameters: 94,410,389
Creating datasets...
Testing data loading...


/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_499233/4089412068.py:143: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Sample batch - Input shape: torch.Size([2, 106752]), Labels shape: torch.Size([2, 69])
Input values stats: min=-5.3669, max=7.7348, mean=0.0000


/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Tra

Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 39.59 GiB total capacity; 1.80 GiB already allocated; 5.19 MiB free; 1.90 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
tokens = row["phonemes"].split()

labels = []
for t in tokens:
    token_id = processor.tokenizer.convert_tokens_to_ids(t)
    if token_id is None:
        print("UNKNOWN TOKEN:", repr(t))
        print("Unicode:", [hex(ord(c)) for c in t])
        raise ValueError(f"Token {t} not in vocab")
    labels.append(token_id)
